# t-stack-trajectory — worked example 2: Build trajectory by stacking generator outputs

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `t-stack-trajectory`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A common pattern in generative modelling is to record the latent vector at every denoising or diffusion step. Each call to the model produces one latent of shape `(D,)`. After all steps, `torch.stack` assembles the list into a `(T, D)` trajectory tensor suitable for plotting or downstream analysis.

## Worked solution

**Step 1 — Collect per-step latents.**
Initialize an empty list. In each iteration of the denoising loop, call the model, detach the output, and append the resulting tensor.

**Step 2 — Stack at the end.**
After the loop, `t.stack(latents, dim=0)` turns the list of T tensors each of shape `(D,)` into a `(T, D)` matrix.

**Step 3 — Why not cat?**
`torch.cat` would need each element to already be 2D (`(1, D)`). Stacking is cleaner because the new axis is inserted for you.

**Step 4 — Verify the shape.**
Assert that `trajectory.shape == (T, D)` before using it downstream, especially if T can be 0 (empty trajectory edge case).

In [ ]:
import torch as t

t.manual_seed(0)
D = 8   # latent dimension
T = 10  # denoising steps

def fake_denoise_step(z: t.Tensor, step: int) -> t.Tensor:
    """Toy model: reduce magnitude slightly at each step."""
    return z * 0.9 + 0.1 * t.randn_like(z)

# Initialize from noise
z = t.randn(D)
latents = [z.clone()]  # include the initial noise as step 0

for step in range(T):
    z = fake_denoise_step(z, step)
    latents.append(z.detach().clone())

# Stack into trajectory
traj = t.stack(latents, dim=0)   # (T+1, D)
print('Steps collected:', len(latents))      # T+1
print('Trajectory shape:', traj.shape)       # (11, 8)
print('Initial z norm:', traj[0].norm().item())
print('Final z norm:', traj[-1].norm().item())